# PaperTrail quickstart
Where did the 87.3% in Table 4 come from? PaperTrail links every number in your paper to the experiment
run that produced it, and flags numbers that went stale after a rerun.

If it helps you, please ⭐ the repo: https://github.com/Anandharajan/papertrail

In [ ]:
!pip install -q "papertrail[mlflow] @ git+https://github.com/Anandharajan/papertrail"

## 1. Log results from your experiment (plain Python, or MLflow / W&B / DVC)

In [ ]:
import os, mlflow
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("demo")
with mlflow.start_run(run_name="baseline") as run:
    mlflow.log_param("lr", 1e-3)
    mlflow.log_metric("acc_test", 0.8731)
    mlflow.log_metric("f1_macro", 0.9121)
!papertrail import mlflow sqlite:///mlflow.db --experiment demo --results results.json

## 2. Scan a paper: which numbers come from which run?

In [ ]:
os.makedirs("paper", exist_ok=True)
open("paper/main.tex", "w").write(r"""Our model reaches 87.31\% test accuracy and a macro F1 of 0.912.
An unexplained 42.17 appears here.""")
!papertrail scan paper/ --results results.json | head -40
!papertrail freeze paper/ --results results.json

## 3. Rerun the experiment and catch the stale number

In [ ]:
with mlflow.start_run(run_name="baseline-rerun"):
    mlflow.log_metric("acc_test", 0.8802)
    mlflow.log_metric("f1_macro", 0.9121)
!papertrail import mlflow sqlite:///mlflow.db --run {mlflow.search_runs(order_by=["start_time DESC"]).run_id[0]} --results results.json
!papertrail check paper/ --results results.json

Issues and ideas: https://github.com/Anandharajan/papertrail/issues